# Week 5 · Notebook 2 — Classification
**CSCI 250 · Fall 2026**

Predict a **category**. We train **logistic regression, k-NN, and a decision tree** on a two-class dataset, read **precision / recall / F1 / confusion matrix**, and run an **overfitting study** on tree depth.

> Runs in Google Colab. No API keys needed. Part of **Assignment A5**.

## 0. Install

In [ ]:
!pip -q install scikit-learn matplotlib

## 1. Make a classification dataset
`make_classification` builds a synthetic, slightly **imbalanced** problem — perfect for showing why accuracy alone can mislead.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    weights=[0.8, 0.2], random_state=42)   # 80/20 class imbalance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
import numpy as np
print('class balance (train):', np.bincount(y_train))

## 2. Train three classifiers
Identical API; only the estimator class changes (Week 4's big idea).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

models = {
    'logreg': LogisticRegression(max_iter=1000),
    'knn   ': KNeighborsClassifier(n_neighbors=5),
    'tree  ': DecisionTreeClassifier(max_depth=5, random_state=42),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, '-> accuracy', round(m.score(X_test, y_test), 3))

## 3. Why accuracy isn't enough
With 80/20 classes, even a lazy model scores ~0.8. **Precision/recall/F1** tell the real story. `classification_report` prints all three per class.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best = models['logreg']
preds = best.predict(X_test)
print(classification_report(y_test, preds, digits=3))

## 4. The confusion matrix
Rows = actual, columns = predicted. The off-diagonal cells are exactly your mistakes (false positives and false negatives).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, preds)
print(cm)
ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap='Blues')
plt.title('Logistic regression — confusion matrix'); plt.show()

## 5. Overfitting study: decision-tree depth
A deeper tree fits training data better — until it starts **memorizing noise**. Watch the **train↔test gap** open up. That gap is the signature of overfitting.

In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_acc, 'o-', label='train')
plt.plot(depths, test_acc,  's-', label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.title('Decision tree: train vs test accuracy')
plt.legend(); plt.grid(alpha=0.3); plt.show()

Where the two curves **diverge** is where overfitting begins: train keeps climbing toward 1.0 while test plateaus or drops. The best `max_depth` is usually near the peak of the **test** curve — that is regularization by limiting complexity.

## 6. Cross-validation: a more stable score
A single split can be lucky. `cross_val_score` averages over 5 folds for a steadier estimate.

In [ ]:
from sklearn.model_selection import cross_val_score

for name, m in models.items():
    s = cross_val_score(m, X_train, y_train, cv=5, scoring='f1')
    print(f'{name}  F1 = {s.mean():.3f} +/- {s.std():.3f}')

---
## A5 — Part 2 tasks
1. Print a `classification_report` and confusion matrix for **all three** models.
2. State which model wins by **F1** (not accuracy) and why F1 is the fairer metric here.
3. From the overfitting plot, give the `max_depth` where overfitting begins.
4. Write a 200-word analysis (combine with Part 1) for your A5 submission. Add an **AI Use** note.

In [ ]:
# Your A5 Part 2 work here
